# Feature Engineering with Shapley Values for Cluster Analysis
### Load clustering results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import gc
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import shap
warnings.filterwarnings('ignore')

with open('data/clustering_results.pkl', 'rb') as f:
    clustering_results = pickle.load(f)

cluster_df = clustering_results['cluster_df']
print(f"Loaded patients: {len(cluster_df)}")
print(f"Cluster distribution:")
print(cluster_df['cluster'].value_counts().sort_index())

# Load prepared time series data
with open('data/ts_prepared_data.pkl', 'rb') as f:
    ts_data = pickle.load(f)

# Load final ML dataset
ml_df = pd.read_csv('data/ml_dataset_ready.csv')
print(f"\nLoaded ML features: {ml_df.shape}")

# Create mapping dictionary for patient_id
ml_df['patient_id'] = ml_df['patient_id'].astype(str)
cluster_df['patient_id'] = cluster_df['patient_id'].astype(str)

# Merge with clusters
ml_with_clusters = ml_df.merge(
    cluster_df[['patient_id', 'cluster']], 
    on='patient_id', 
    how='inner'
)

print(f"\nAfter merging: {len(ml_with_clusters)} patients")
print(f"Cluster distribution in ML dataset:")
print(ml_with_clusters['cluster'].value_counts().sort_index())

### CREATE ADDITIONAL FEATURES

In [ ]:
# Load daily statistics for more detailed features
daily_stats = ts_data['daily_stats']

# Create features for each patient
patient_features = []

for patient_id in ml_with_clusters['patient_id'].unique():
    patient_data = daily_stats[daily_stats['id пациента'].astype(str) == patient_id]
    
    if len(patient_data) > 0:
        features = {'patient_id': patient_id}
        
        # 1. Blood pressure variability (clinically important feature)
        features['sbp_variability'] = patient_data['САД'].std()
        features['dbp_variability'] = patient_data['ДАД'].std()
        features['pp_variability'] = patient_data['пульсовое_давление'].std() if 'пульсовое_давление' in patient_data.columns else (patient_data['САД'] - patient_data['ДАД']).std()
        
        # 2. Rate of BP change (trends)
        if len(patient_data) >= 3:
            days = np.arange(len(patient_data))
            slope_sbp = np.polyfit(days, patient_data['САД'].values, 1)[0]
            features['sbp_trend'] = slope_sbp
            
            slope_dbp = np.polyfit(days, patient_data['ДАД'].values, 1)[0]
            features['dbp_trend'] = slope_dbp
        else:
            features['sbp_trend'] = 0
            features['dbp_trend'] = 0
        
        # 3. Percentage of time with elevated BP (SBP > 140 or DBP > 90)
        high_bp = ((patient_data['САД'] > 140) | (patient_data['ДАД'] > 90)).mean()
        features['pct_hypertension'] = high_bp * 100
        
        # 4. Percentage of time with low BP (SBP < 90 or DBP < 60)
        low_bp = ((patient_data['САД'] < 90) | (patient_data['ДАД'] < 60)).mean()
        features['pct_hypotension'] = low_bp * 100
        
        # 5. Coefficient of variation (stability)
        features['sbp_cv'] = (patient_data['САД'].std() / patient_data['САД'].mean()) * 100 if patient_data['САД'].mean() > 0 else 0
        features['dbp_cv'] = (patient_data['ДАД'].std() / patient_data['ДАД'].mean()) * 100 if patient_data['ДАД'].mean() > 0 else 0
        
        # 6. Extreme values
        features['sbp_extreme_high_pct'] = (patient_data['САД'] > 180).mean() * 100
        features['sbp_extreme_low_pct'] = (patient_data['САД'] < 100).mean() * 100
        
        patient_features.append(features)

# Create dataframe with new features
additional_features = pd.DataFrame(patient_features)

print(f"\nCreated additional features: {len(additional_features.columns) - 1}")
print(f"Additional features:\n{additional_features.columns.tolist()}")

# Merge with main ML dataset
ml_with_clusters = ml_with_clusters.merge(additional_features, on='patient_id', how='left')

print(f"\nFinal dataset: {ml_with_clusters.shape}")

### DATA PREPARATION

In [ ]:
# Define target variable
target = ml_with_clusters['cluster'].values

# TEMPORAL FEATURES - EXCLUDED (already accounted for in GDTW!)
temporal_features = ['время измерения', 'время измерения_days_diff']

# EXCLUDE OBSERVATION GROUP AND TIMEZONE FROM PLOTS
exclude_from_plots = ['группа наблюдения', 'часовой пояс']

# Define all features for the model
exclude_cols = ['patient_id', 'cluster', 'kzs_count', 'target_high_risk', 'target_risk_category'] + temporal_features
feature_cols = [col for col in ml_with_clusters.columns if col not in exclude_cols]
for feat in temporal_features:
    print(f"  - {feat}")
for feat in exclude_from_plots:
    print(f"  - {feat}")
for feat in feature_cols[:15]:
    print(f"  - {feat}")

# Create copy for modeling
df_model = ml_with_clusters[feature_cols].copy()

In [ ]:
# Encode categorical features
from sklearn.preprocessing import LabelEncoder

for col in df_model.columns:
    if df_model[col].dtype == 'object' or df_model[col].dtype == 'str' or df_model[col].dtype.name == 'string':
        print(f"\nEncoding feature: {col}")
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))

non_numeric = []
for col in df_model.columns:
    if not pd.api.types.is_numeric_dtype(df_model[col]):
        non_numeric.append(col)
        print(f"  x {col}: {df_model[col].dtype}")
    else:
        print(f"  ✓ {col}: {df_model[col].dtype}")

if non_numeric:
    for col in non_numeric:
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')
        print(f"  Forcibly converted {col}")
else:
    print("\nAll columns are numeric!")

# Handle missing values
print("\nHandling missing values...")
for col in df_model.columns:
    if df_model[col].isnull().any():
        median_val = df_model[col].median()
        df_model[col].fillna(median_val, inplace=True)
        print(f"  {col}: filled with median {median_val:.2f}")

print(f"\nMissing values after processing: {df_model.isnull().sum().sum()}")

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model)
X = pd.DataFrame(X_scaled, columns=df_model.columns)
y = target

print(f"\nFeature matrix: {X.shape}")
print(f"Cluster distribution: {np.bincount(y)}")

### MODEL TRAINING

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Model training
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Evaluation
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[f'Cluster_{i}' for i in range(1, 4)]))

# Cross-validation
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')
print(f"\nCross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': df_model.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTOP-15 IMPORTANT FEATURES")
for idx, row in feature_importance.head(15).iterrows():
    print(f"  {row['feature']:<30}: {row['importance']:.4f}")

### SHAP ANALYSIS

In [ ]:
# Alternative version with separator lines between features
fig, ax = plt.subplots(figsize=(15, 8))

n_clusters = len(cluster_data)
bar_width = 0.25
x = np.arange(n_features)

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
cluster_labels = ['Cluster 1', 'Cluster 2', 'Cluster 3']

# Draw bars
for idx, shap_df in enumerate(cluster_data):
    cluster_shap = []
    for feature in features:
        shap_val = shap_df[shap_df['feature'] == feature]['mean_shap'].values
        cluster_shap.append(shap_val[0] if len(shap_val) > 0 else 0)
    
    offset = (idx - n_clusters/2 + 0.5) * bar_width
    bars = ax.bar(x + offset, cluster_shap, bar_width, 
                   label=cluster_labels[idx], 
                   color=colors[idx], alpha=0.85, 
                   edgecolor='black', linewidth=0.5)
    
    # Add values on bars
    for bar, val in zip(bars, cluster_shap):
        if abs(val) > 0.01:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., 
                       height + 0.02,
                       f'{val:.2f}', ha='center', va='bottom',
                       fontsize=8, rotation=0, fontweight='bold')
            else:
                ax.text(bar.get_x() + bar.get_width()/2., 
                       height - 0.02,
                       f'{val:.2f}', ha='center', va='top',
                       fontsize=8, rotation=0, fontweight='bold')

# Add vertical separator lines between features
for i in range(1, n_features):
    ax.axvline(x=i - 0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Add semi-transparent backgrounds for groups (alternating)
for i in range(n_features):
    if i % 2 == 0:  # Alternate for better readability
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.05, color='lightblue', zorder=0)

# Configure labels
feature_names_en = [feature_translation.get(f, f) for f in features]
ax.set_xticks(x)
ax.set_xticklabels(feature_names_en, fontsize=10, rotation=45, ha='right')
ax.set_ylabel('Mean SHAP Value', fontsize=12)
ax.set_xlabel('Features', fontsize=12)
ax.set_title('Feature Importance Comparison Across Clusters\n(Positive = pushes toward cluster, Negative = pushes away)', 
             fontsize=13, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Annotations
ax.annotate('▲ Positive SHAP', xy=(0.02, 0.98), xycoords='axes fraction',
            fontsize=9, color='green', va='top', bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))
ax.annotate('▼ Negative SHAP', xy=(0.02, 0.94), xycoords='axes fraction',
            fontsize=9, color='red', va='top', bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('data/photo/shap_cluster_importance_with_lines.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: data/photo/shap_cluster_importance_with_lines.png")

### FEATURE IMPORTANCE COMPARISON BY CLUSTER

In [ ]:
# Analysis for each cluster
cluster_interpretation = {}

for class_idx in range(n_classes):
    print(f"\n{'='*50}")
    print(f"CLUSTER {class_idx + 1}")
    print(f"{'='*50}")
    
    shap_values_class = shap_values[:, :, class_idx]
    mean_shap = np.mean(shap_values_class, axis=0)
    
    # Create dataframe and exclude unnecessary features
    shap_df = pd.DataFrame({
        'feature': df_model.columns,
        'mean_shap': mean_shap
    })
    shap_df = shap_df[~shap_df['feature'].isin(exclude_from_plots)]
    
    # Top-5 positive (increase probability)
    positive = shap_df[shap_df['mean_shap'] > 0].sort_values('mean_shap', ascending=False).head(5)
    # Top-5 negative (decrease probability)
    negative = shap_df[shap_df['mean_shap'] < 0].sort_values('mean_shap', ascending=True).head(5)
    
    print(f"\nFeatures that increase probability:")
    for _, row in positive.iterrows():
        print(f"  + {row['feature']}: {row['mean_shap']:.4f}")
    
    print(f"\nFeatures that decrease probability:")
    for _, row in negative.iterrows():
        print(f"  - {row['feature']}: {row['mean_shap']:.4f}")
    
    cluster_interpretation[class_idx + 1] = {
        'increasing': positive['feature'].tolist(),
        'decreasing': negative['feature'].tolist()
    }

### CLINICAL INTERPRETATION

In [ ]:
print("\nCLUSTER 1 (HIGH RISK):")
print("   Characteristics:")
if 1 in cluster_interpretation:
    inc = cluster_interpretation[1]['increasing']
    dec = cluster_interpretation[1]['decreasing']
    if 'sbp_trend' in inc:
        print("   - Positive SBP trend (pressure increasing)")
    if 'dbp_trend' in inc:
        print("   - Positive DBP trend")
    if 'pct_hypertension' in inc:
        print("   - High percentage of time with hypertension")
    if 'sbp_variability' in inc:
        print("   - High blood pressure variability")
    print("\n   Clinical recommendations:")
    print("   - Require most intensive monitoring")
    print("   - Antihypertensive therapy adjustment needed")
    print("   - Increased risk of cardiovascular complications")

print("\nCLUSTER 2 (MODERATE RISK):")
print("   Characteristics:")
if 2 in cluster_interpretation:
    inc = cluster_interpretation[2]['increasing']
    dec = cluster_interpretation[2]['decreasing']
    if 'sbp_trend' in dec:
        print("   - Negative SBP trend (pressure decreasing)")
    if 'dbp_trend' in dec:
        print("   - Negative DBP trend")
    if 'pct_hypertension' in dec:
        print("   - Moderate percentage of time with hypertension")
    print("\n   Clinical recommendations:")
    print("   - Regular monitoring (once every 2-3 months)")
    print("   - Maintain current therapy")
    print("   - Risk factor control")

print("\nCLUSTER 3 (LOW RISK):")
print("   Characteristics:")
if 3 in cluster_interpretation:
    inc = cluster_interpretation[3]['increasing']
    dec = cluster_interpretation[3]['decreasing']
    if 'sbp_trend' in dec:
        print("   - Stable blood pressure readings")
    if 'pct_hypertension' in dec:
        print("   - Low percentage of time with hypertension")
    print("\n   Clinical recommendations:")
    print("   - Routine observation (once every 6 months)")
    print("   - Preventive measures")
    print("   - Healthy lifestyle education programs")

# Save feature importance
feature_importance.to_csv('data/feature_importance_clean.csv', index=False)
shap_importance.to_csv('data/shap_importance_clean.csv', index=False)

# Save full report
ml_with_clusters['predicted_cluster'] = rf_model.predict(X)
ml_with_clusters['cluster_1_prob'] = rf_model.predict_proba(X)[:, 0]
ml_with_clusters['cluster_2_prob'] = rf_model.predict_proba(X)[:, 1]
ml_with_clusters['cluster_3_prob'] = rf_model.predict_proba(X)[:, 2]

ml_with_clusters.to_csv('data/clinical_report_clean.csv', index=False)
print("Saved: data/clinical_report_clean.csv")

print("\nCREATED FILES:")
print("  - data/clinical_report_clean.csv - full clinical report")
print("  - data/feature_importance_clean.csv - feature importance (Gini)")
print("  - data/shap_importance_clean.csv - feature importance (SHAP)")
print("\nCREATED PLOTS:")
print("  - data/photo/shap_cluster_importance_clean.png - importance by cluster")

# Clear memory
gc.collect()